# CodeGuide-LLM - Colab 训练配置

本笔记本提供在 Google Colab 上训练 CodeGuide-LLM 的完整配置方案。

## 环境要求
- GPU：NVIDIA A100 (40GB) / L4 (24GB) / T4 (16GB)
- Python：3.11
- CUDA：12.1
- 磁盘空间：≥ 50 GB

## 步骤 1：环境配置与依赖安装

In [ ]:
# 检查 GPU 状态
!nvidia-smi

# 检查 Python 版本
!python --version

# 检查 CUDA 版本
!nvcc --version

In [ ]:
# 安装依赖
!pip install -q ipywidgets==8.1.5

# 安装 PyTorch 2.4.0 + CUDA 12.1
!pip install -q torch==2.4.0 torchvision==0.19.0 torchaudio==2.4.0 --index-url https://download.pytorch.org/whl/cu121

# 安装项目依赖
!pip install -q unsloth[cu121-ampere-torch240]>=2024.12.0
!pip install -q trl>=0.12.0
!pip install -q transformers>=4.47.0
!pip install -q datasets>=3.0.0
!pip install -q peft>=0.14.0
!pip install -q accelerate>=1.1.0
!pip install -q bitsandbytes>=0.45.0
!pip install -q flash-attn>=2.7.0
!pip install -q einops>=0.8.0
!pip install -q openai>=1.55.0
!pip install -q wandb>=0.18.0
!pip install -q numpy>=1.26.0
!pip install -q pandas>=2.2.0
!pip install -q tqdm>=4.66.0
!pip install -q jsonlines>=4.0.0
!pip install -q pytest>=8.0.0
!pip install -q rich>=13.9.0
!pip install -q omegaconf>=2.3.0

print("✅ 依赖安装完成！")

## 步骤 2：挂载 Google Drive（可选，推荐）

In [ ]:
from google.colab import drive
import os

# 挂载 Drive
drive.mount('/content/drive')

# 创建工作目录
WORK_DIR = '/content/drive/MyDrive/CodeGuide-LLM'
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)

print(f"✅ 工作目录已设置：{WORK_DIR}")

## 步骤 3：克隆项目代码

In [ ]:
# 方式 1：从 GitHub 克隆（如果你已经推送到 GitHub）
!git clone https://github.com/你的用户名/CodeGuide-LLM.git .

# 方式 2：如果是本地项目，使用文件上传功能
# 或者直接在 Colab 中创建文件结构

print("✅ 项目代码已获取！")

## 步骤 4：配置 API Keys

In [ ]:
import os
from getpass import getpass

# 设置 OpenAI API Key（用于蒸馏和评测）
print("请输入 OpenAI API Key：")
os.environ["OPENAI_API_KEY"] = getpass()

# 配置 WandB（实验追踪）
!wandb login

print("✅ API Keys 已配置！")

## 步骤 5：数据准备

In [ ]:
# 如果需要运行蒸馏
!python scripts/build_sft_dataset.py \
    --max_items 1000 \
    --concurrency 5 \
    --quality_threshold 0.6 \
    --out data/sft_train.jsonl

# 或者直接使用已有的数据（如果存在）
print("✅ 数据准备完成！")

## 步骤 6：修改 Colab 专用配置文件

In [ ]:
# 创建 Colab 专用配置文件
import yaml

colab_config = {
    "model": {
        "base_model": "Qwen/Qwen2.5-Coder-7B-Instruct",
        "max_seq_length": 4096,
        "quantization": "nf4",
        "lora_rank": 16,
        "lora_alpha": 32,
        "lora_dropout": 0.05
    },
    "training": {
        "output_dir": "models/grpo_final",
        "num_train_epochs": 3,
        "per_device_train_batch_size": 2,
        "per_device_eval_batch_size": 2,
        "gradient_accumulation_steps": 8,
        "learning_rate": 1e-5,
        "warmup_ratio": 0.03,
        "max_grad_norm": 0.3,
        "save_steps": 100,
        "eval_steps": 100,
        "logging_steps": 10,
        "fp16": True,
        "bf16": True,
        "save_best": True,
        "best_model_dir": "models/grpo_best"
    },
    "grpo": {
        "num_generations": 4,
        "max_new_tokens": 1024,
        "reward_weights": {
            "accuracy": 0.6,
            "format": 0.4,
            "teaching": 0.0
        },
        "normalize_rewards": True
    },
    "curriculum": {
        "enabled": False,
        "stages": [
            {"name": "easy", "difficulty": "easy", "max_new_tokens": 512},
            {"name": "medium", "difficulty": "medium", "max_new_tokens": 768},
            {"name": "hard", "difficulty": "hard", "max_new_tokens": 1024}
        ]
    },
    "data": {
        "train_file": "data/sft_train.jsonl",
        "eval_file": "data/eval.jsonl",
        "test_size": 0.1
    },
    "wandb": {
        "project": "codeguide-llm",
        "name": "colab-run-1",
        "log_model": True
    }
}

with open('configs/colab_config.yaml', 'w', encoding='utf-8') as f:
    yaml.dump(colab_config, f, default_flow_style=False, allow_unicode=True)

print("✅ Colab 配置文件已创建！")

## 步骤 7：开始训练

In [ ]:
# 先运行 SFT 训练（可选热启动）
!python scripts/train_sft.py --config configs/colab_config.yaml

# 然后运行 GRPO 训练
!python src/training/grpo_train.py --config configs/colab_config.yaml

## 步骤 8：推理 Demo

In [ ]:
# CLI 推理
!python scripts/inference_demo.py --model models/grpo_best/

## Colab 使用技巧

### 1. 防止断开连接
在浏览器控制台中运行：
```javascript
function ClickConnect() {
  console.log("Working");
  document.querySelector("colab-toolbar-button#connect").click();
}
setInterval(ClickConnect, 60000);
```

### 2. 检查磁盘空间
!df -h

### 3. 监控 GPU 使用率
!watch -n 1 nvidia-smi

### 4. Colab Pro 资源
- L4 GPU (24GB)：适合 7B 模型
- A100 GPU (40GB)：适合更大模型或更快训练
- T4 GPU (16GB)：适合轻量实验

### 5. 保存模型到 Hugging Face Hub
```python
from huggingface_hub import notebook_login
notebook_login()
model.push_to_hub("你的用户名/CodeGuide-LLM")
```